# Allele frequency comparison: 1000G reference vs AoU

Pulls AoU's own allele frequencies (not the reference's) for the exact variant set a projection actually used, then compares against the reference's frequencies for the same variants -- a systematic frequency divergence (points off the y=x line, not just noise) would point to a real batch/harmonization issue between ACAF and 1000G, independent of the ID/allele-matching checks already done. Defaults to round 2's paths; change `REF_ACOUNT_PATH`/`VARIANT_LIST_PATH`/etc. to point at round 1's instead.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Inputs

Defaults are round 2's paths (`submit_pca_r2.ipynb`'s `CLUSTER = "ceu_gbr"` output). `VARIANT_LIST_PATH` should be `list-variants` output (the exact set a projection actually used), not a theoretical intersection -- same lesson as everywhere else in this pipeline.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"
R2_OUT_DIR = f"{BUCKET_DIR}/round2_pca"

CLUSTER = "ceu_gbr"

ACAF_BIALLELIC_PREFIX = f"{BUCKET_DIR}/round1_pca/acaf_hm3_subset_biallelic"
ROUND1_KEEP_PATH = f"{BUCKET_DIR}/round1_pca/round1_keep_ids_eur_p99.9.txt"

REF_ACOUNT_PATH = f"{R2_OUT_DIR}/r2_pca_{CLUSTER}.acount"                       # reference frequencies, from submit_pca_r2.ipynb
VARIANT_LIST_PATH = f"{R2_OUT_DIR}/r2_projected_{CLUSTER}.sscore.vars"          # list-variants output -- the exact set actually scored

OUT_DIR = R2_OUT_DIR
AOU_FREQ_PREFIX = os.path.join(OUT_DIR, f"aou_freq_{CLUSTER}")

os.makedirs(OUT_DIR, exist_ok=True)

## Compute AoU's own allele frequencies

Same sample restriction (`ROUND1_KEEP_PATH`) and variant set (`VARIANT_LIST_PATH`) as the projection itself -- `--nonfounders` for the same reason as everywhere else in this pipeline (AoU's psam has no real pedigree, so "founder" status is meaningless here).

In [ ]:
%%bash -s "$ACAF_BIALLELIC_PREFIX" "$ROUND1_KEEP_PATH" "$VARIANT_LIST_PATH" "$AOU_FREQ_PREFIX"
set -e
ACAF_BIALLELIC_PREFIX=$1
ROUND1_KEEP_PATH=$2
VARIANT_LIST_PATH=$3
AOU_FREQ_PREFIX=$4

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$ROUND1_KEEP_PATH" \
  --extract "$VARIANT_LIST_PATH" \
  --nonfounders \
  --freq counts \
  --out "$AOU_FREQ_PREFIX"

wc -l "${AOU_FREQ_PREFIX}.acount"

## Load both frequency files and merge

`.acount` columns: `#CHROM ID REF ALT PROVISIONAL_REF? ALT_CTS OBS_CT` -- ALT allele frequency is `ALT_CTS / OBS_CT`. Merge on ID -- both files were built with the same `--set-all-var-ids`/biallelic-filtering scheme upstream, so ID alone is a safe join key here (already confirmed exact ID+REF+ALT agreement for this variant set in the harmonization diagnostics).

In [ ]:
ref_af = pd.read_csv(REF_ACOUNT_PATH, sep="\t")
ref_af["ref_freq"] = ref_af["ALT_CTS"] / ref_af["OBS_CT"]

aou_af = pd.read_csv(f"{AOU_FREQ_PREFIX}.acount", sep="\t")
aou_af["aou_freq"] = aou_af["ALT_CTS"] / aou_af["OBS_CT"]

merged = ref_af[["ID", "REF", "ALT", "ref_freq"]].merge(
    aou_af[["ID", "REF", "ALT", "aou_freq"]], on="ID", suffixes=("_ref", "_aou")
)
assert (merged["REF_ref"] == merged["REF_aou"]).all() and (merged["ALT_ref"] == merged["ALT_aou"]).all(), \
    "REF/ALT disagree between the two frequency files for some merged IDs -- investigate before trusting the comparison"

print(f"{len(merged)} variants with frequencies in both datasets")
merged[["ref_freq", "aou_freq"]].describe()

## Compare

Scatter with the y=x line -- real, systematic divergence (a shifted cloud, not just scatter around the diagonal) would suggest a batch/harmonization issue distinct from the ID/allele-matching already checked. `abs_diff` flags the biggest individual outliers for closer inspection.

In [ ]:
corr = merged["ref_freq"].corr(merged["aou_freq"])
merged["abs_diff"] = (merged["ref_freq"] - merged["aou_freq"]).abs()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(merged["ref_freq"], merged["aou_freq"], s=3, alpha=0.2, rasterized=True)
ax.plot([0, 1], [0, 1], color="red", linestyle="--", linewidth=1, label="y = x")
ax.set_xlabel("Reference (1000G) ALT allele frequency")
ax.set_ylabel("AoU ALT allele frequency")
ax.set_title(f"Allele frequency comparison -- r = {corr:.4f}, n = {len(merged)}")
ax.legend()
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, f"allele_freq_comparison_{CLUSTER}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

print(f"\nCorrelation: r = {corr:.4f}")
print("Largest frequency differences:")
print(merged.sort_values("abs_diff", ascending=False)[["ID", "REF_ref", "ALT_ref", "ref_freq", "aou_freq", "abs_diff"]].head(20))